# fenics-pipeline — End-to-End Testing Guide

This guide walks you through running the pipeline for the first time, from a fresh clone to a finished STL file. Follow each step in order. Every section tells you what you should see when things are working, and what to do if they are not.

---

## Before you begin

You will need the following on your Windows machine before starting:

- **Docker Desktop** installed and running, with the WSL2 backend enabled
- **WSL2** installed (Windows Subsystem for Linux 2)
- **Git** to clone the repository
- A terminal application — Windows Terminal is recommended

You do not need Python, FEniCSx, or any other software installed locally. Everything runs inside Docker.

---

## Part 1 — First-time machine setup

This section only needs to be done once per machine. If you have already done this, skip to Part 2.

### Step 1.1 — Clone the repository

Open a WSL2 terminal and run:

```bash
git clone <your-repo-url> fenics-pipeline
cd fenics-pipeline
```

### Step 1.2 — Configure WSL2 memory

FEniCSx and gmsh require more memory than WSL2 allocates by default. This step increases the limit.

Open a **Windows PowerShell** window (not WSL2 — a regular PowerShell terminal on Windows) and navigate to the repository folder:

```powershell
cd \\wsl$\Ubuntu\home\<your-username>\fenics-pipeline
```

Then run:

```powershell
powershell.exe -ExecutionPolicy Bypass -File scripts/setup_wsl.ps1
```

You should see output like this:

```
Written to C:\Users\<you>\.wslconfig
Contents:
[wsl2]
memory=24GB
processors=10
swap=8GB

Shutting down WSL2 to apply config...
WSL2 memory now reported as: 23Gi
Done. Start your WSL2 terminal and run: make build && make up
```

If your machine has less than 24GB of RAM, pass a lower value:

```powershell
.\scripts\setup_wsl.ps1 -MemoryGB 16 -Processors 8 -SwapGB 8
```

After this script finishes, **close your PowerShell window and reopen your WSL2 terminal**. WSL2 has restarted with the new memory limits.

### Step 1.3 — Verify the environment

Back in your WSL2 terminal, from the `fenics-pipeline` directory:

```bash
make check
```

You should see output like this:

```
── fenics-pipeline environment check ──

  ✓ WSL2 memory: 23GB
  ✓ Docker daemon reachable
  ✓ Docker Compose: 2.x.x
  ✓ /dev/shm: 1.0GB
  ✓ .env present
       NOTEBOOK_DIR=./notebooks
  ✓ notebooks/: 7 .ipynb files found
  ✓ outputs/meshes/
  ✓ outputs/stl/
  ✓ outputs/reports/
  ✓ outputs/executed_nbs/

✅ All checks passed — run: make build && make up
```

If any line shows `✗`, follow the instruction printed next to it before continuing.

---

## Part 2 — Build the Docker image

This step downloads and compiles the full software stack inside Docker. It takes approximately 10–20 minutes the first time. Subsequent builds are much faster because Docker caches the layers.

```bash
make build
```

You will see a stream of output as Docker downloads base images and installs packages. It is finished when you see:

```
 => exporting to image
 => => writing image sha256:...
 => => naming to docker.io/library/fenics-pipeline:0.8.0
```

If the build fails, the most common cause is a network timeout during package download. Run `make build` again — it will resume from where it left off.

---

## Part 3 — Start JupyterLab

```bash
make up
```

You should see:

```
[+] Running 1/1
 ✔ Container fenics-pipeline  Started
JupyterLab: http://localhost:8888
```

Open your browser and go to `http://localhost:8888`. You should see the JupyterLab interface with the `notebooks/` directory listed on the left.

If the browser shows a connection error, wait 10 seconds and refresh — the container may still be starting up. If it still does not load after 30 seconds, run:

```bash
docker logs fenics-pipeline
```

and look for any error messages near the bottom.

---

## Part 4 — Run the environment validation notebook

This notebook checks that all software dependencies are correctly installed inside the container. Always run this first on a new machine.

In JupyterLab, open `00_env_validation.ipynb`. At the top of the interface, make sure the kernel shown in the top-right corner reads **FEniCSx Pipeline**. If it shows something else, click it and select **FEniCSx Pipeline** from the dropdown.

Run all cells by clicking **Run → Run All Cells** from the menu.

The final cell should print:

```
  dolfinx              0.8.0  ✓
  gmsh                 4.12.2 ✓
  pyvista              0.43.10 ✓
  trimesh              4.4.0  ✓
  skimage              0.22.0 ✓
  papermill            2.6.0  ✓

  MPI comm size: 1
  MPI rank:      0
  OpenSCAD:      OpenSCAD version 2021.01
  PyVista headless render: OK

✅ Environment validated — safe to proceed to 01_geometry_openscad.ipynb
```

If any package shows `MISSING` or a version mismatch, stop here and re-run `make build` from your WSL2 terminal. Then restart the container with `make up` and try again.

---

## Part 5 — Run the pipeline interactively, one stage at a time

Running each stage notebook individually lets you inspect the outputs at every step before committing to the next. This is the recommended approach for your first run.

For each notebook below, open it in JupyterLab and click **Run → Run All Cells**. Then check the expected output described before moving to the next notebook.

---

### Stage 1 — `01_geometry_openscad.ipynb`

**What it does:** Reads parameters from `scad/params.json`, compiles `scad/base_part.scad` using OpenSCAD, and produces an STL file of the part geometry.

**Run it:** Open `01_geometry_openscad.ipynb` and run all cells.

**What you should see at the end of Cell 3:**

```
Success:    True
Duration:   3.2s
STL path:   outputs/meshes/base_part.stl
STL size:   142.6 KB
```

**What you should see at the end of Cell 4:**

```
Vertices:  4820
Faces:     9636
Bounds:    [0.0, 100.0, 0.0, 60.0, 0.0, 20.0]

Preview saved: outputs/reports/base_part_geometry.png
```

**Check the output:** In JupyterLab, navigate to `outputs/reports/` in the left panel and open `base_part_geometry.png`. You should see a 3D render of a rectangular bracket with four mounting holes.

**What you should see at the end of Cell 5:**

```
Handoff written: outputs/meshes/base_part_stage01.json
```

**If something goes wrong:**
- `Success: False` with `stderr: openscad binary not found` — OpenSCAD did not install correctly in the container. Run `make build` again.
- `Success: False` with geometry errors in stderr — there is a problem with `scad/base_part.scad`. Check that the file was saved correctly.
- STL size is 0 KB — OpenSCAD ran but produced empty output. This usually means a parameter in `scad/params.json` is invalid, such as a `wall_thickness` larger than half the part width.

---

### Stage 2 — `02_mesh_gmsh.ipynb`

**What it does:** Takes the STL from Stage 1, imports it into gmsh, generates a volumetric tetrahedral mesh, checks mesh quality, and exports to XDMF format for FEniCSx.

**Run it:** Open `02_mesh_gmsh.ipynb` and run all cells.

**What you should see at the end of Cell 2:**

```
Success:          True
Duration:         45.2s
Nodes:            18432
Elements:         82156
Boundary tris:    6240
MSH:              outputs/meshes/base_part.msh
XDMF:             outputs/meshes/base_part.xdmf
```

The element count will vary depending on your machine and the `target_element_size` in `scad/params.json`. Anything between 20,000 and 200,000 elements is normal.

**What you should see at the end of Cell 3:**

```
Elements:          82156
Aspect ratio:      mean=2.14  max=8.32  p95=4.21
Min dihedral:      18.4°
Inverted elements: 0
Passed:            True
```

The critical values are `Inverted elements: 0` and `Passed: True`. If either of these is wrong, do not proceed to Stage 3.

**Check the output:** Open `outputs/reports/base_part_mesh.png`. You should see the tetrahedral mesh overlaid on the part geometry. Open `outputs/reports/base_part_aspect_ratio.png` to see the element quality distribution — the bulk of the histogram should be left of the orange warning line.

**If something goes wrong:**
- `No surfaces found after STL classification` — the STL from Stage 1 has geometry errors. Go back to Stage 1 and check `outputs/meshes/base_part.stl` opens correctly in a viewer.
- `Inverted elements: N` where N > 0 — the mesh has degenerate elements. Reduce `target_element_size` in `scad/params.json` by half and re-run Stage 2.
- `Passed: False` with aspect ratio failures — try changing `ALGORITHM_3D` in Cell 0 from `4` to `1` and re-running.
- Meshing takes more than 5 minutes — the `target_element_size` in `scad/params.json` is too small. Increase it from `2.0` to `4.0` for a faster test mesh.

---

### Stage 3 — `03_fea_fenicsx.ipynb`

**What it does:** Loads the mesh from Stage 2, applies boundary conditions (fixed bottom face, load on top face), and solves the linear elasticity equations to compute displacement and stress fields.

**What to expect:** This is the most computationally intensive stage. On a typical machine it takes 1–5 minutes depending on mesh size.

**Run it:** Open `03_fea_fenicsx.ipynb` and run all cells.

**What you should see at the end of Cell 3:**

```
Success:          True
Duration:         87.4s
DOFs:             55,296
Max displacement: 0.0412 mm
Von Mises max:    184.3 MPa
Von Mises mean:   22.1 MPa
```

The exact numbers will differ based on your mesh and parameters. What matters is that all values are positive and physically reasonable — displacement in the range of 0.001–10mm for a steel part under 1000N is expected.

**What you should see at the end of Cell 4:**

```
Yield strength:   250.0 MPa
Von Mises max:    184.3 MPa
Safety factor:    1.36

⚠ Safety factor < 1.5 — marginal. Optimization will reduce it further.
```

A safety factor warning here is normal for the default parameters — it means the optimizer will have meaningful work to do. If the safety factor is below 1.0 (the part yields before optimization), go back to `scad/params.json` and either increase `wall_thickness` or reduce `load_magnitude_n`.

**Check the output:** Open `outputs/reports/base_part_stress.png`. You should see a colour map of the part where red/orange areas indicate high stress (typically around the mounting holes and load application area) and blue areas indicate low stress (candidates for material removal).

**If something goes wrong:**
- `ModuleNotFoundError: dolfinx` — the kernel subprocess is not finding the FEniCSx environment. Restart the kernel from the JupyterLab menu (Kernel → Restart Kernel) and run all cells again.
- The solve hangs with no output for more than 10 minutes — MPI is likely stuck. Stop the cell, restart the kernel, and try setting `USE_ITERATIVE_SOLVER = True` in Cell 0.
- `Segmentation fault` in the PyVista cell — set `PYVISTA_OFF_SCREEN=true` in `docker-compose.yml` environment block and restart with `make up`.

---

### Stage 4 — `04_simp_optimization.ipynb`

**What it does:** Runs the SIMP topology optimization loop. Starting from a uniform density field, it iteratively removes material from low-stress regions while maintaining structural stiffness, converging to a binary solid/void distribution.

**What to expect:** This stage runs between 20 and 100 iterations. Each iteration solves the full FEA problem with the current density field. At the default mesh size and 100 iterations, expect 20–60 minutes. You will see a line of output per iteration as it runs.

**Run it:** Open `04_simp_optimization.ipynb` and run all cells.

**What you should see during Cell 3** (one line per iteration):

```
  Iter    1 | C=4.2831e+03 | Vol=0.400 | Δρ=0.1842 | 18.3s
  Iter    2 | C=3.1204e+03 | Vol=0.400 | Δρ=0.1421 | 17.8s
  Iter    3 | C=2.8831e+03 | Vol=0.400 | Δρ=0.1102 | 18.1s
  ...
  Iter   47 | C=1.9204e+03 | Vol=0.400 | Δρ=0.0009 | 17.4s

✓ Converged at iteration 47 (Δρ=9.2e-04 < 1e-03)
```

The compliance value (`C=`) should decrease and then flatten. Volume fraction (`Vol=`) should stay close to your target. The change in density field (`Δρ=`) should trend toward zero.

**What you should see at the end of Cell 3:**

```
Success:          True
Converged:        True
Iterations:       47
Final compliance: 1.9204e+03
Final vol frac:   0.401
Duration:         834.2s (13.9 min)
```

**Check the output:** Open `outputs/reports/base_part_density_final.png`. The left panel shows the full continuous density field (grey = void, white = solid). The right panel shows the thresholded result — you should see clear structural members connecting the load application point to the mounting holes, with material removed from low-stress interior regions.

Open `outputs/reports/base_part_convergence.png`. The compliance curve should show a sharp initial drop followed by a gradual flattening. If it has not flattened by the last iteration, go back to Cell 0 and increase `MAX_ITERATIONS` to 150 and re-run Cell 3 only.

**If something goes wrong:**
- `Baseline safety factor X is already below SAFETY_FACTOR_MIN` — the part fails before optimization. Go back to Stage 3 and check the safety factor output. Adjust geometry or load parameters.
- Compliance does not decrease after the first few iterations — `PENAL` may be too low. Try increasing it from `3.0` to `3.5` in Cell 0.
- STL in Stage 5 looks like a solid block — `VOLUME_FRACTION` is too high. Try reducing it from `0.4` to `0.3`.
- Optimization runs all 100 iterations without converging — this is not a failure. The result is still usable. Increase `MAX_ITERATIONS` if you want to push further.

---

### Stage 5 — `05_stl_export.ipynb`

**What it does:** Converts the continuous density field from Stage 4 into a watertight STL file. It voxelizes the density field onto a regular grid, extracts the isosurface using marching cubes, repairs the mesh, and exports.

**Run it:** Open `05_stl_export.ipynb` and run all cells.

**What you should see at the end of Cell 4:**

```
Running marching cubes at threshold 0.5...
Vertices:  48320
Triangles: 96640
Bounds:    X=[2.1, 97.9] Y=[1.8, 58.2] Z=[0.0, 20.0] mm
```

**What you should see at the end of Cell 5:**

```
After merge:  24160 verts, 96640 faces
No disconnected islands found
Applied 5 Laplacian smoothing iterations

Final: 24160 verts, 96640 faces
Watertight: True
Volume:     28441.32 mm³
Area:       14820.16 mm²
```

The critical value is `Watertight: True`. A watertight mesh can be sent directly to a 3D printer slicer. If it reads `False`, see the troubleshooting section below.

**What you should see at the end of Cell 6:**

```
STL exported:  outputs/stl/base_part_optimized.stl
File size:     4821.4 KB
Triangles:     96640
Watertight:    True
Volume:        28441.32 mm³
BB volume:     120000.00 mm³
Fill ratio:    23.7% (target was 40%)
```

The fill ratio being lower than the target volume fraction is normal — the threshold at 0.5 cuts more aggressively than the raw volume fraction.

**Check the output:** Open `outputs/reports/base_part_before_after.png`. The left panel shows the original solid bracket; the right panel shows the topology-optimized version with material removed from low-stress regions. You should see clear structural members — arching load paths from the top face to the mounting hole regions — rather than a solid block.

**If something goes wrong:**
- `Marching cubes produced no geometry` — lower `DENSITY_THRESHOLD` from `0.5` to `0.4` in Cell 0.
- `Watertight: False` — increase `VOXEL_RESOLUTION` from `100` to `150` in Cell 0 and re-run from Cell 3. Alternatively increase `SMOOTH_ITERATIONS` from `5` to `10`.
- STL has spikes or disconnected floating pieces — go back to `04_simp_optimization.ipynb` Cell 0, increase `FILTER_RADIUS` from `6.0` to `9.0`, and re-run the optimization from Cell 3.
- Fill ratio is much higher than expected — the density field did not converge to a binary distribution. Increase `PENAL` in Stage 4 and re-run the optimization.

---

## Part 6 — Run the test suite

The test suite verifies the core modules independently of the pipeline. It builds its own minimal meshes internally and does not require any pipeline outputs.

From your WSL2 terminal:

```bash
make test
```

You should see output like this:

```
tests/test_mesh_quality.py::TestAspectRatio::test_equilateral_tet_aspect_ratio_near_one PASSED
tests/test_mesh_quality.py::TestAspectRatio::test_needle_tet_high_aspect_ratio PASSED
tests/test_mesh_quality.py::TestAspectRatio::test_multiple_tets PASSED
tests/test_mesh_quality.py::TestInvertedDetection::test_no_inverted_in_regular_tet PASSED
tests/test_mesh_quality.py::TestInvertedDetection::test_detects_inverted_tet PASSED
tests/test_mesh_quality.py::TestQualityReport::test_good_mesh_passes_default_thresholds PASSED
tests/test_mesh_quality.py::TestQualityReport::test_good_mesh_aspect_ratio_reasonable PASSED
tests/test_mesh_quality.py::TestQualityReport::test_thin_mesh_fails_strict_thresholds PASSED
tests/test_mesh_quality.py::TestQualityReport::test_quality_report_summary_is_string PASSED
tests/test_mesh_quality.py::TestQualityReport::test_gmsh_not_left_initialized PASSED
tests/test_fea_smoke.py::TestImportChain::test_dolfinx_imports PASSED
tests/test_fea_smoke.py::TestImportChain::test_mpi4py_functional PASSED
tests/test_fea_smoke.py::TestImportChain::test_petsc4py_imports PASSED
tests/test_fea_smoke.py::TestImportChain::test_ufl_imports PASSED
tests/test_fea_smoke.py::TestImportChain::test_src_fea_imports PASSED
tests/test_fea_smoke.py::TestImportChain::test_src_solver_imports PASSED
tests/test_fea_smoke.py::TestBoundaryConditions::test_dirichlet_bc_nonzero_dofs PASSED
tests/test_fea_smoke.py::TestBoundaryConditions::test_traction_tag_matches_load_hints PASSED
tests/test_fea_smoke.py::TestFEASolve::test_trivial_solve_completes PASSED
tests/test_fea_smoke.py::TestFEASolve::test_solve_result_physically_plausible PASSED
tests/test_fea_smoke.py::TestFEASolve::test_higher_load_gives_higher_stress PASSED

21 passed in 94.3s
```

All 21 tests should pass. If any fail, the test output will tell you exactly which assertion failed and what values were produced, which points to the specific module that needs attention.

---

## Part 7 — Run the full pipeline headlessly

Once all five stage notebooks have run successfully interactively, you can run the entire pipeline in one command. This is how you would run it in production or on a schedule.

First clean the existing outputs so the headless run starts fresh:

```bash
make clean-outputs
```

Then run:

```bash
make run
```

Papermill will execute all five stage notebooks in sequence. You will see progress output in the terminal as each stage completes. The full run will take approximately the same time as running all five notebooks manually.

When it finishes successfully you will see:

```
  ✓ 01_geometry_openscad.ipynb          3.2s
  ✓ 02_mesh_gmsh.ipynb                  45.2s
  ✓ 03_fea_fenicsx.ipynb                87.4s
  ✓ 04_simp_optimization.ipynb          834.2s
  ✓ 05_stl_export.ipynb                 12.1s
  STL: outputs/stl/base_part_optimized.stl

✅ All runs completed successfully
```

The executed notebooks — with all cell outputs preserved — are saved to `outputs/executed_nbs/`. Open any of them in JupyterLab to inspect what happened at each step.

If a stage fails, the terminal output will show which notebook failed and which cell raised the error. Open the corresponding executed notebook in `outputs/executed_nbs/` to see the full traceback.

---

## Part 8 — Collect your outputs

After a successful run, your outputs are:

| File | Description |
|---|---|
| `outputs/stl/base_part_optimized.stl` | The final topology-optimized part, ready for 3D printing or further CAD work |
| `outputs/reports/base_part_before_after.png` | Side-by-side comparison of original and optimized geometry |
| `outputs/reports/base_part_stress.png` | Von Mises stress map from the FEA solve |
| `outputs/reports/base_part_convergence.png` | SIMP optimization convergence history |
| `outputs/reports/base_part_density_final.png` | Final density field before and after thresholding |
| `outputs/executed_nbs/` | Full executed notebooks with all outputs, timestamped |

The STL file can be opened in any 3D printing slicer (PrusaSlicer, Cura, Bambu Studio) or CAD tool (Fusion 360, FreeCAD) for inspection and further processing.

---

## Quick reference — common commands

```bash
# First-time setup
make check          # validate environment before building
make build          # build Docker image (~10 min)
make up             # start JupyterLab on http://localhost:8888

# Daily use
make run            # run full pipeline headlessly
make test           # run test suite
make clean-outputs  # clear all generated files

# Troubleshooting
docker logs fenics-pipeline          # container logs
docker exec fenics-pipeline python -c "import dolfinx; print(dolfinx.__version__)"
make down && make up                 # restart the container
```